# Create BHF Awards (British Heart Foundation)

Creates BHF awards parsed from the BHF annual Research Grant Award report PDFs.

**Prerequisites:** run `scripts/local/bhf_to_s3.py` to download + parse + upload first.

**Data source:** bhf.org.uk per-year Research Grant Award report PDFs (2004/5 through 2024/25), linked from `/for-professionals/information-for-researchers/previous-awards`. Each PDF lists awards under scheme sections; every row carries a real BHF grant reference + PI + institution + grant title + GBP amount + duration. PDFs have a text layer (not scanned) — parsed via pypdf + regex.

**S3 location:** `s3a://openalex-ingest/awards/bhf/bhf_projects.parquet`

**BHF funder in OpenAlex:** funder_id 4320319992 · display_name "British Heart Foundation" · GB.

**Schema notes:**
- `funder_award_id` = the BHF grant reference (e.g. `FS/IBSRF/24/25222`, `PG/25/12329`, `RG/20/2/34763`) — **this is the form cited in BHF-funded papers** (high work-linkage; the value of this ingest).
- `amount` = GBP. The parser requires valid thousands-grouping so a glued page-number can't extend the amount (a 2016-PDF artifact `£1,415,37914` → `£1,415,379`). Max legit = £30M (CureHeart Big Beat Challenge); median ~£232K.
- `pi_given_name` = the PI's initials (BHF PDFs publish initials + surname, not full first names); `pi_family_name` = surname (degree post-nominals stripped: BSc/MD/PhD/FAHA/FMedSci/...).
- `institution_name` = UK research institution (86%; the ~14% misses are wrapped/abbreviated institution names in older PDFs).
- `start_year` = the award-report year.

**⚠️ §6.4a NOTE — high-frequency PI combos are REAL, not a scraper bug.** Because BHF PIs are identified by initial+surname over a 20-year span, the §6.4a frequency check shows some combos with n>5: e.g. (M, Bennett)=17 is Prof Martin Bennett (Cambridge, cardiovascular — 17 genuine grants across 2004-2024, diverse titles, all University of Cambridge), (M, Mayr)=16 is Prof Manuel Mayr (King's). These are prolific cardiovascular researchers legitimately holding many grants, NOT a DOM-node bug. Verified by inspecting their grant titles/institutions. Do NOT NULL them.

provenance `bhf_grant_pdfs`, priority 211. **Coverage caveat:** 7 of the ~20 PDFs (mostly 2005-2014) use a layout pypdf extracts differently and yielded 0 rows — documented as a follow-up; the 2,346 parsed cover 2004/5, 2008/9, and 2016-2024/25 in full.

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.bhf_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/bhf/bhf_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.bhf_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.bhf_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.bhf_raw LIMIT 5;

## Step 1.6: Funder existence fail-fast

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi FROM openalex.common.funder WHERE funder_id = 4320319992;

## Step 2: Create BHF Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.bhf_awards
USING delta
AS
WITH
bhf_funder AS (
    SELECT funder_id, display_name, ror_id, doi FROM openalex.common.funder WHERE funder_id = 4320319992
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        g.title as display_name,
        CAST(NULL AS STRING) as description,
        f.funder_id,
        g.funder_award_id as funder_award_id,
        CASE WHEN TRY_CAST(g.amount AS DOUBLE) > 0 THEN TRY_CAST(g.amount AS DOUBLE) ELSE NULL END as amount,
        CASE WHEN TRY_CAST(g.amount AS DOUBLE) > 0 THEN 'GBP' ELSE NULL END as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        CASE
            WHEN g.funder_award_id RLIKE '^(FS|SP/PhD|DJT)' THEN 'fellowship'
            WHEN g.funder_award_id RLIKE '^(CH|SI|RE|AA)' THEN 'research'
            ELSE 'research'
        END as funding_type,
        CAST(NULL AS STRING) as funder_scheme,
        'bhf_grant_pdfs' as provenance,
        CASE WHEN TRY_CAST(g.start_year AS INT) IS NOT NULL
             THEN CAST(CONCAT(g.start_year, '-01-01') AS DATE) ELSE NULL END as start_date,
        CAST(NULL AS DATE) as end_date,
        TRY_CAST(g.start_year AS INT) as start_year,
        CAST(NULL AS INT) as end_year,
        CASE
            WHEN g.pi_family_name IS NOT NULL OR g.institution_name IS NOT NULL THEN
                struct(
                    g.pi_given_name as given_name,
                    g.pi_family_name as family_name,
                    CAST(NULL AS STRING) as orcid,
                    struct(
                        g.institution_name as name,
                        'United Kingdom' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation,
                    CAST(NULL AS DATE) as role_start
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>, role_start:DATE
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>, role_start:DATE
        >>) as investigators,
        'https://www.bhf.org.uk/for-professionals/information-for-researchers/previous-awards' as landing_page_url,
        CAST(NULL AS STRING) as doi,
        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.bhf_raw g
    CROSS JOIN bhf_funder f
    WHERE g.funder_award_id IS NOT NULL AND TRIM(CAST(g.funder_award_id AS STRING)) != ''
)
SELECT * FROM awards_transformed;

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'bhf_grant_pdfs' AND priority = 211;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    211 as priority
FROM openalex.awards.bhf_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_bhf_awards FROM openalex.awards.bhf_awards;

In [ ]:
%sql
SELECT funder_award_id, display_name, amount, currency, start_year, lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.bhf_awards LIMIT 10;

In [ ]:
%sql
-- §6.4a freq check. NOTE: top combos with n>5 are REAL prolific cardiovascular PIs (Bennett/Mayr/Poole), NOT a scraper bug — BHF PIs are initial+surname over a 20-year span. Verified by title/institution diversity. Do not NULL.
SELECT lead_investigator.given_name AS g, lead_investigator.family_name AS f, COUNT(*) AS n
FROM openalex.awards.bhf_awards GROUP BY 1,2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT
    COUNT(*) as total, COUNT(amount) as has_amount, COUNT(lead_investigator.family_name) as has_pi,
    COUNT(lead_investigator.affiliation.name) as has_inst, COUNT(start_date) as has_start,
    ROUND(try_divide(COUNT(amount) * 100.0, COUNT(*)), 1) as pct_amount,
    ROUND(try_divide(COUNT(lead_investigator.affiliation.name) * 100.0, COUNT(*)), 1) as pct_inst
FROM openalex.awards.bhf_awards;

In [ ]:
%sql
SELECT start_year, COUNT(*) FROM openalex.awards.bhf_awards WHERE start_year IS NOT NULL GROUP BY 1 ORDER BY 1 DESC LIMIT 25;

In [ ]:
%sql
SELECT lead_investigator.affiliation.name as inst, COUNT(*) as n
FROM openalex.awards.bhf_awards WHERE lead_investigator.affiliation.name IS NOT NULL
GROUP BY 1 ORDER BY n DESC LIMIT 20;